# Local true latent MSE under the raw-MSE detours

Use a GPU runtime. This notebook reopens the two goals and the same three trajectories per goal from the detour audit. It renders a small grid around each set of paths, encodes it with the original frozen encoder, and overlays the real paths. It also re-encodes the captured frames along those paths and scores them against the exact goal tensors saved by the collector.

These are costs on **real images**. The world-model planner used predicted terminal latents to choose a sequence of actions. The overlay can reveal misleading regions in the real cost surface and identify cases where the executed path goes uphill in true cost.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import io
import json
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/experience_temporal_half_square_v1')
RUN_ID='20261107T150425413490Z'  # The run used in the attached audit.
SUCCESS_RADIUS=4.5
MOVE_TOLERANCE=1e-5

In [ ]:
def read_json(path):
    return json.loads(path.read_text())

def find_run(root, run_id):
    candidates=[root/run_id] if run_id else sorted(root.iterdir(),reverse=True)
    for candidate in candidates:
        status=candidate/'stages/band_1.json'
        if not status.is_file():
            continue
        stage=read_json(status)
        if stage['status']=='complete' and stage.get('condition',{}).get('collector')=='raw':
            return candidate
    raise ValueError('No completed raw-MSE band 1 found. Check RUNS and RUN_ID.')

def load_paths(run):
    stage=read_json(run/'stages/band_1.json')
    manifest=read_json(run/'experiment.json')['manifest']
    cases=manifest['stages'][0]
    expected=hashlib.sha256(json.dumps(cases,sort_keys=True,allow_nan=False).encode()).hexdigest()
    if stage['cases_digest']!=expected or stage['episodes']!=len(cases):
        raise ValueError('Band 1 cases differ from the saved manifest.')
    goals=sorted({tuple(case['goal']) for case in cases},key=lambda goal:(goal[1],goal[0]))
    if len(goals)<2 or goals[0][1]==goals[-1][1]:
        raise ValueError('Band 1 must contain upper and lower goals.')
    result={}
    for label,goal in [('Upper goal',goals[0]),('Lower goal',goals[-1])]:
        rows=[]
        for case in cases:
            if tuple(case['goal'])!=goal:
                continue
            case_id=case['case_id']
            record=read_json(run/f'episodes/band_1/{case_id}.json')
            if (record['case']!=case or record['runtime_id']!=stage['runtime_id']
                    or record['condition']!=stage['condition']):
                raise ValueError(f'Episode provenance differs for {case_id}.')
            raw_path=(run/record['raw_path']).resolve()
            raw_path.relative_to(run.resolve())
            payload=raw_path.read_bytes()
            if hashlib.sha256(payload).hexdigest()!=record['raw_sha256']:
                raise ValueError(f'Raw observations changed for {case_id}.')
            raw=torch.load(io.BytesIO(payload),map_location='cpu',weights_only=True)
            xy=raw['states'][:,:2].numpy().astype(float)
            steps=record['micro_steps']
            if (raw['case']!=case or raw['runtime_id']!=stage['runtime_id']
                    or raw['success']!=record['success'] or len(xy)!=steps+1
                    or not np.array_equal(raw['primitive_steps'].numpy(),np.arange(steps+1))
                    or not np.isfinite(xy).all()):
                raise ValueError(f'Raw observations differ from the record for {case_id}.')
            rows.append(dict(case=case_id,goal=goal,success=record['success'],xy=xy,
                             frames=raw['frames'],goal_index=case['goal_index'],
                             mpc_updates=record['mpc_updates'],start=case['start'],
                             seed=case['seed'],layout=case['layout']))
        result[label]=rows
    return result

run=find_run(RUNS,RUN_ID)
paths=load_paths(run)
print('Run ID:',run.name)
print('Collector:',read_json(run/'stages/band_1.json')['condition']['collector'])
for label,rows in paths.items():
    print(label,rows[0]['goal'],'·',len(rows),'rollouts')

In [ ]:
def measure_path(row):
    xy=row['xy']
    distance=np.linalg.norm(xy-row['goal'],axis=1)
    change=np.diff(distance)
    length=np.linalg.norm(np.diff(xy,axis=0),axis=1).sum()
    displacement=np.linalg.norm(xy[-1]-xy[0])
    return dict(case=row['case'],success=row['success'],actions=len(change),
                initial_distance=distance[0],final_distance=distance[-1],path_length=length,
                extra_travel=max(0.,length-displacement),away_steps=int((change>MOVE_TOLERANCE).sum()),
                away_distance=np.maximum(change,0).sum(),
                worst_backtrack=(distance-np.minimum.accumulate(distance)).max())

def choose_three(rows, stats):
    # Use successful rollouts for the visual comparison when at least three exist.
    by_case={r['case']:r for r in rows}
    pool=[s for s in stats if s['success']]
    if len(pool)<3:
        pool=stats
    ordered=sorted(pool,key=lambda s:(s['extra_travel'],s['case']))
    indices=np.linspace(0,len(ordered)-1,min(3,len(ordered)),dtype=int)
    return [by_case[ordered[i]['case']] for i in indices]


shown={}
for label,rows in paths.items():
    shown[label]=choose_three(rows,[measure_path(r) for r in rows])
    print(label,':',[r['case'] for r in shown[label]])

## Load the original encoder and goal tensors

The notebook checks the goal file against the checksum saved before band 1. It prints the old and current numerical runtime and measures the MSE of a fresh goal rendering against the saved target. The world model is unnecessary for this audit.

In [ ]:
import re
import shutil
import subprocess
import sys
from types import SimpleNamespace

assert torch.cuda.is_available(), 'Select a GPU runtime.'
runtime=read_json(run/'runtime.json')
signature=runtime['signature']
stage=read_json(run/'stages/band_1.json')
assert runtime['runtime_id']==stage['runtime_id'], 'Band 1 runtime changed.'

def checkout(url,path,commit,branch=None):
    assert re.fullmatch(r'[a-f0-9]{40}',commit), 'Invalid recorded commit.'
    if not path.exists():
        args=['git','clone']
        if branch:
            args+=['--branch',branch,'--single-branch']
        subprocess.run([*args,url,str(path)],check=True)
    assert subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True).strip()==url
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
    exists=subprocess.run(['git','-C',str(path),'cat-file','-e',commit],capture_output=True)
    if exists.returncode:
        subprocess.run(['git','-C',str(path),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(path),'checkout','--detach',commit],check=True)

REPO=Path('/content/i-jepa-wm-raw-mse-audit')
DINO=Path('/content/dino_wm')
checkout('https://github.com/kmylonas/i-jepa-wm.git',REPO,runtime['source_commit'],'curriculum-cost-adapter')
checkout('https://github.com/gaoyuezhou/dino_wm.git',DINO,signature['dino_commit'])
subprocess.run([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml',
                'scipy','einops','pillow','imageio','imageio-ffmpeg'],check=True)
sys.path.insert(0,str(REPO))

from src.mylonas_ijepa_wm.curriculum_runtime import PilotRuntime,file_sha256,source_digest
from src.mylonas_ijepa_wm.environment_adapters import WallAdapter
from src.mylonas_ijepa_wm.environment_configs import EnvironmentConfig
from src.mylonas_ijepa_wm.plan_pointmaze import load_frozen_ijepa_encoder

assert source_digest(REPO/'src')==signature['source'], 'Encoder source differs from collection.'
assert source_digest(DINO/'env/wall')==signature['dino_source'], 'Renderer source differs from collection.'
goal_path=run/'collection_goals.pt'
assert file_sha256(goal_path)==stage['condition']['goal_sha256'], 'Saved collector goal tokens changed.'
targets=torch.load(goal_path,map_location='cpu',weights_only=True)
assert targets['runtime_id']==runtime['runtime_id'], 'Goal tensors belong to another runtime.'
target_tokens=targets['tokens']

device=torch.device('cuda')
variant=signature['runtime_variant']
torch.set_float32_matmul_precision(variant['matmul_precision'])
print('Collection GPU:',variant['device_name'],'· Torch:',signature['torch_version'])
print('Audit GPU:',torch.cuda.get_device_name(device),'· Torch:',str(torch.__version__))
expected=signature['models']['encoder']
encoder_path=Path('/content/raw-mse-audit-encoder.pth.tar')
if not encoder_path.is_file() or file_sha256(encoder_path)!=expected:
    shutil.copy2('/content/drive/MyDrive/checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar',encoder_path)
assert file_sha256(encoder_path)==expected, 'Encoder checkpoint differs from collection.'
encoder=load_frozen_ijepa_encoder(encoder_path,device)
encoding=SimpleNamespace(device=device,precision=signature['precision'],
    args=SimpleNamespace(encoder_batch_size=signature['encoder_batch_size']),load_encoder=lambda:encoder)
env=WallAdapter(EnvironmentConfig(env='wall',wall_layouts=[[32,30]],dino_wm_root=str(DINO)),100)

## Score the real frames and local grid

The grid covers the full extent of the three observed trajectories and their goal. A move must change physical distance by at least 0.1 units to count in the directional checks. The cost threshold is 0.1% of the local map's MSE range, with a small minimum for numerical noise.

In [ ]:
GRID_SPACING=0.5
DISTANCE_GAP=0.1

def costs_for_frames(frames,goal_index):
    tokens=PilotRuntime.encode_frames(encoding,frames)
    return (tokens-target_tokens[goal_index]).square().mean(dim=(1,2)).numpy()

def axis_for(values,low,high):
    lo=np.floor((min(values)-1)/GRID_SPACING)*GRID_SPACING
    hi=np.ceil((max(values)+1)/GRID_SPACING)*GRID_SPACING
    return np.unique(np.clip(np.arange(lo,hi+GRID_SPACING/2,GRID_SPACING),low,high))

def grid_guidance(distance,cost,tol):
    directed=[]
    for axis in (0,1):
        first=np.take(distance,range(distance.shape[axis]-1),axis=axis)
        last=np.take(distance,range(1,distance.shape[axis]),axis=axis)
        delta=last-first
        change=np.diff(cost,axis=axis)*-np.sign(delta)
        mask=(np.abs(delta)>=DISTANCE_GAP)&(np.maximum(first,last)>SUCCESS_RADIUS)
        directed.extend(change[mask])
    directed=np.asarray(directed)
    return dict(toward_edges=len(directed),cost_decreases=int((directed < -tol).sum()),
                cost_increases=int((directed > tol).sum()),ties=int((np.abs(directed)<=tol).sum()))

maps={}
grid_rows=[]
path_rows=[]
for label,rows in shown.items():
    reference=rows[0]
    goal=reference['goal']
    goal_index=reference['goal_index']
    env.reset(reference['seed'],layout=reference['layout'],start=reference['start'],goal=goal)
    fresh=costs_for_frames(np.stack([env.render_goal()]),goal_index)[0]
    points=np.vstack([*(r['xy'] for r in rows),np.asarray(goal)-SUCCESS_RADIUS,np.asarray(goal)+SUCCESS_RADIUS])
    xs=axis_for(points[:,0],35.001,59.999)
    ys=axis_for(points[:,1],4.001,59.999)
    gx,gy=np.meshgrid(xs,ys)
    xy=np.column_stack([gx.ravel(),gy.ravel()])
    scores=[]
    for offset in range(0,len(xy),encoding.args.encoder_batch_size):
        frames=np.stack([env._render_at(p) for p in xy[offset:offset+encoding.args.encoder_batch_size]])
        scores.extend(costs_for_frames(frames,goal_index))
    cost=np.asarray(scores).reshape(gx.shape)
    distance=np.hypot(gx-goal[0],gy-goal[1])
    tol=max(1e-7,.001*float(np.ptp(cost)))
    maps[label]=dict(xs=xs,ys=ys,cost=cost,goal=goal)
    grid_rows.append(dict(group=label,fresh_goal_mse=fresh,**grid_guidance(distance,cost,tol)))
    for row in rows:
        row['true_mse']=costs_for_frames(row['frames'],row['goal_index'])
        row['distance']=np.linalg.norm(row['xy']-goal,axis=1)
        change=np.diff(row['true_mse'])
        delta=np.diff(row['distance'])
        away=delta>=DISTANCE_GAP
        toward=delta<=-DISTANCE_GAP
        path_rows.append(dict(group=label,case=row['case'],mpc_updates=row['mpc_updates'],
            start_mse=row['true_mse'][0],end_mse=row['true_mse'][-1],
            away_steps=int(away.sum()),away_and_cost_drops=int((away&(change < -tol)).sum()),
            toward_steps=int(toward.sum()),toward_and_cost_rises=int((toward&(change > tol)).sum())))

print('Local grid edges moving toward the goal from outside the success circle')
display(pd.DataFrame(grid_rows).round(6))
print('Exact captured frames along the six paths')
display(pd.DataFrame(path_rows).round(6))

## Overlay the three paths per goal

Open circles mark starts, crosses mark endpoints, and the star marks the goal. Solid curves in the lower panels are true latent MSE on the recorded images. Dashed curves are physical distance. The same colors identify each episode throughout.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as effects
from matplotlib.patches import Circle

fig,axes=plt.subplots(2,2,figsize=(14,10),constrained_layout=True)
for column,(label,rows) in enumerate(shown.items()):
    spatial,trace=axes[:,column]
    distance_axis=trace.twinx()
    grid=maps[label]
    goal=grid['goal']
    image=spatial.pcolormesh(grid['xs'],grid['ys'],grid['cost'],shading='nearest',cmap='viridis',vmin=0)
    fig.colorbar(image,ax=spatial,label='True raw latent MSE')
    spatial.add_patch(Circle(goal,SUCCESS_RADIUS,fill=False,edgecolor='white',ls=':'))
    spatial.scatter(*goal,marker='*',c='white',edgecolor='black',s=125,zorder=5)
    for color,row in zip(('tab:blue','tab:orange','tab:green'),rows):
        xy=row['xy']
        tag=f"{row['case']} · {measure_path(row)['extra_travel']:.1f} extra"
        spatial.plot(*xy.T,color=color,marker='.',markersize=3,label=tag,
                     path_effects=[effects.Stroke(linewidth=3,foreground='white'),effects.Normal()])
        spatial.scatter(*xy[0],s=60,facecolors='white',edgecolors=color,zorder=4)
        spatial.scatter(*xy[-1],s=40,marker='x',color='white',zorder=4)
        trace.plot(row['true_mse'],color=color,marker='.',markersize=3)
        distance_axis.plot(row['distance'],color=color,ls='--',alpha=.55)
    spatial.invert_yaxis()
    spatial.set(aspect='equal',xlabel='Ball x',ylabel='Ball y',title=f'{label} {tuple(round(v,2) for v in goal)}')
    spatial.legend(loc='best',fontsize=8)
    trace.set(xlabel='Executed action',ylabel='True raw latent MSE')
    distance_axis.set_ylabel('Distance to goal (units)')
    trace.grid(alpha=.2)
fig.savefig('/content/raw_mse_detour_heatmaps.png',dpi=150,bbox_inches='tight')
plt.show()